# Part B3 — Baseline versus Adapter B

The three fixed Part A prompts are run through the fine-tuned model and compared side by side with the Part A baseline. The brief asks what **improved or degraded** in terms of correctness, domain terminology, completeness and hallucinations. All logic is in `src/adapter_eval.py`; this notebook calls it, shows the outputs, and records the analysis.

In [1]:
# Dependencies come from requirements.txt, the single source of truth for every notebook in this repo.
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "adapter_eval.py").is_file():
    ROOT = ROOT.parent
%pip install -q -r {ROOT}/requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, json, html
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import HTML, display
import adapter_eval as ev

pd.set_option("display.width", 220, "display.max_colwidth", 80)
print(ev.check_design())

{'design_check_passed': True, 'prompts': ['malaria_uncomplicated_pf', 'sepsis_iv_antibiotics', 'hypertension_first_line'], 'rubric_quotes_found': 6}


## Method

- **Same settings as the baseline:** greedy decoding, `max_new_tokens=150`, no system prompt, the same `### Instruction` / `### Response` block, the same three prompts from Part A Step 2.
- **Primary comparison: bf16 base + adapter.** The baseline ran in bfloat16, so using the same dtype means the only thing that changed is the fine-tuning. The adapter was *trained* on a 4-bit base, so a **4-bit base + adapter** run is reported next to it as the matching-condition check.
- **The baseline is regenerated in the same session** (before the adapter is attached) and compared with the saved Part A outputs, which proves the Part A baseline reproduces. Tokens per second for baseline and adapter are measured in the same process after a warm-up generation, so they are comparable. The adapter is attached unmerged, which costs some speed.
- **Scoring.** For each prompt a short rubric lists the facts a correct answer contains. Each fact is a pattern, and the rubric is tied to the corpus by exact quotes from the prompt's own corpus file, which `check_design()` verifies. Numbers in an output that appear neither in the prompt nor in that corpus file are listed as ungrounded. A rubric pattern match is a screening tool; the written analysis below reads the outputs themselves.

In [3]:
rows = []
for p in ev.BASELINE_PROMPTS:
    for fid, label, _ in ev.RUBRIC[p["id"]]["facts"]:
        rows.append({"prompt": p["id"], "fact": label, "corpus_file": p["corpus_target"].split("/")[-1]})
print(pd.DataFrame(rows).to_string(index=False))
print()
for p in ev.BASELINE_PROMPTS:
    print(p["id"], "| corpus quotes that anchor the rubric:")
    for q in ev.RUBRIC[p["id"]]["quotes"]:
        print("   -", q)

                  prompt                                                               fact                 corpus_file
malaria_uncomplicated_pf                  names artemisinin-based combination therapy (ACT)  who_malaria_2024-11-30.txt
malaria_uncomplicated_pf                                            artemether-lumefantrine  who_malaria_2024-11-30.txt
malaria_uncomplicated_pf                                             artesunate-amodiaquine  who_malaria_2024-11-30.txt
malaria_uncomplicated_pf                                              artesunate-mefloquine  who_malaria_2024-11-30.txt
malaria_uncomplicated_pf                                     dihydroartemisinin-piperaquine  who_malaria_2024-11-30.txt
malaria_uncomplicated_pf                             artesunate + sulfadoxine-pyrimethamine  who_malaria_2024-11-30.txt
malaria_uncomplicated_pf                                            artesunate-pyronaridine  who_malaria_2024-11-30.txt
   sepsis_iv_antibiotics                

## Run (GPU)

Baseline again in bf16, then the adapter on bf16, then the adapter on a 4-bit base.

In [4]:
report = ev.run_eval()
print("baseline reproduces the saved Part A outputs:", report["baseline_matches_part_a_outputs"])
print("peak GPU memory (GB): bf16 run", report["peak_gpu_gb_bf16_run"], "| 4-bit run", report["peak_gpu_gb_4bit_run"])
print("eval loss on the 10 held-out rows, before -> after training (B2):", [round(x, 4) for x in report["eval_loss_before_after_training"]])
print("adapter sha256:", report["adapter_sha256"][:16], "...")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

baseline reproduces the saved Part A outputs: {'malaria_uncomplicated_pf': True, 'sepsis_iv_antibiotics': True, 'hypertension_first_line': True}
peak GPU memory (GB): bf16 run 14.55 | 4-bit run 4.24
eval loss on the 10 held-out rows, before -> after training (B2): [2.1271, 1.4267]
adapter sha256: 868435f70bdfce7e ...


## Side-by-side outputs

Column 1 is the Part A baseline, column 2 the adapter on bf16 (the primary comparison), column 3 the adapter on 4-bit.

In [5]:
by = {s: {r["prompt_id"]: r for r in report["results"][s]} for s in ev.SYSTEMS}
sc = {(s["system"], s["prompt_id"]): s for s in report["scores"]}
cell = "vertical-align:top;padding:6px;border:1px solid #bbb;white-space:pre-wrap;font-size:12px;width:33%"
for p in ev.BASELINE_PROMPTS:
    head = "".join(f"<th style='border:1px solid #bbb;padding:6px'>{n}</th>" for n in ["Baseline (bf16)", "Adapter B on bf16", "Adapter B on 4-bit"])
    tds = ""
    for s in ev.SYSTEMS:
        r, k = by[s][p["id"]], sc[(s, p["id"])]
        meta = f"[{r['new_tokens']} tokens, {'stopped itself' if r['stopped_on_eos'] else 'hit the 150-token limit'}, facts {len(k['facts_hit'])}/{k['facts_total']}]"
        tds += f"<td style='{cell}'><i>{meta}</i>\n\n{html.escape(r['output'])}</td>"
    display(HTML(f"<h4>{p['id']}</h4><p><b>Prompt:</b> {html.escape(p['instruction'])}</p><table style='border-collapse:collapse'><tr>{head}</tr><tr>{tds}</tr></table>"))

Baseline (bf16),Adapter B on bf16,Adapter B on 4-bit
"[150 tokens, hit the 150-token limit, facts 1/7] The malaria guidance recommends artemether-lumefantrine (AL) for uncomplicated Plasmodium falciparum malaria in adults and in children weighing at least 25 kg. ### Evidence: The malaria guidance recommends AL for uncomplicated Plasmodium falciparum malaria in adults and in children weighing at least 25 kg. ### References: 1. World Health Organization. Guidelines for the treatment of malaria. 2015. ### Notes: The malaria guidance recommends AL for uncomplicated Plasmodium falciparum malaria in adults and in children weigh","[73 tokens, stopped itself, facts 2/7] Per WHO malaria guidance, the recommended first-line treatment is artemisinin-based combination therapy with artesunate plus sulfadoxine-pyrimethamine, and the second-line treatment is quinine plus clindamycin. This output is for educational/reference purposes only and must not replace professional clinical judgment.","[79 tokens, stopped itself, facts 2/7] Per WHO malaria guidance, the first-line treatment is artemisinin-based combination therapy (ACT) with artesunate plus sulfadoxine-pyrimethamine (AS+SP), and the second-line treatment is quinine plus clindamycin. This output is for educational/reference purposes only and must not replace professional clinical judgment."


Baseline (bf16),Adapter B on bf16,Adapter B on 4-bit
"[150 tokens, hit the 150-token limit, facts 2/4] In an adult with suspected sepsis, intravenous antibiotics should be started as soon as possible, ideally within 1 hour of recognition of sepsis. If the source of infection is still uncertain, broad-spectrum antibiotics should be started. ### Evidence: The Surviving Sepsis Campaign (SSC) guidelines recommend that intravenous antibiotics should be started as soon as possible, ideally within 1 hour of recognition of sepsis. The SSC guidelines also recommend that broad-spectrum antibiotics should be started if the source of infection is still uncertain. ### References: 1. Surviving","[128 tokens, stopped itself, facts 3/4] Per Sepsis-3 guidelines, antibiotics should be started within 1 hour of recognition of sepsis, and the choice of antibiotics should be based on the most likely source of infection, with broad-spectrum coverage including gram-negative and anaerobic organisms. If the source is still uncertain, empiric coverage should include gram-positive, gram-negative and anaerobic organisms, and the antibiotic regimen should be adjusted as soon as the source is identified. This output is for educational/reference purposes only and must not replace professional clinical judgment.","[107 tokens, stopped itself, facts 2/4] Per SSC sepsis guidelines, antibiotics should be started within 1 hour of recognition of sepsis and the choice of antibiotics depends on local resistance patterns and the most likely source of infection. If the source is still uncertain, broad-spectrum antibiotics should cover gram-negative and gram-positive organisms, including MRSA, and empiric coverage for Pseudomonas is optional. This output is for educational/reference purposes only and must not replace professional clinical judgment."


Baseline (bf16),Adapter B on bf16,Adapter B on 4-bit
"[150 tokens, hit the 150-token limit, facts 2/5] The adult hypertension guidance recommends that the first-line antihypertensive treatment for adults with hypertension is a thiazide-type diuretic, calcium channel blocker, or angiotensin-converting enzyme inhibitor. The choice of first-line antihypertensive treatment is based on the patient's age, comorbidities, and other factors. ### Evidence: The adult hypertension guidance is based on a systematic review of the literature and expert consensus. The review included studies that evaluated the effectiveness and safety of different antihypertensive treatments in adults with hypertension. The guidance recommends a thiazide-type di","[119 tokens, stopped itself, facts 3/5] Per the NICE hypertension guideline, the first-line treatment is a thiazide-like diuretic, with a calcium channel blocker as an alternative, and an ACE inhibitor or ARB as a third option. For people aged 55 or older, a thiazide-like diuretic is the first choice, with a calcium channel blocker as an alternative, and an ACE inhibitor or ARB as a third option. This output is for educational/reference purposes only and must not replace professional clinical judgment.","[119 tokens, stopped itself, facts 3/5] Per the NICE hypertension guideline, the first-line treatment is a thiazide-like diuretic, with an ACE inhibitor or ARB as an alternative, and a calcium channel blocker as a third option. For people aged 55 or older, a thiazide-like diuretic is the first choice, with an ACE inhibitor or ARB as an alternative, and a calcium channel blocker as a third option. This output is for educational/reference purposes only and must not replace professional clinical judgment."


## Scores against the corpus

In [6]:
df = pd.DataFrame(report["scores"])
df["facts"] = df["facts_hit"].apply(len).astype(str) + "/" + df["facts_total"].astype(str)
df["n_flags"] = df["flags"].apply(len)
df["flags"] = df["flags"].apply(lambda x: ", ".join(x) or "-")
df["ungrounded numbers"] = df["ungrounded_numbers"].apply(lambda x: ", ".join(x) or "-")
df["disclaimer"] = df["has_disclaimer"]
show = df[["system", "prompt_id", "facts", "flags", "ungrounded numbers", "disclaimer", "stopped_on_eos", "new_tokens", "extra_sections", "has_references_section"]]
print(show.to_string(index=False))
print()
tot = df.assign(hit=df["facts_hit"].apply(len)).groupby("system", sort=False).agg(facts_hit=("hit", "sum"), facts_total=("facts_total", "sum"),
      flags_raised=("n_flags", "sum"), disclaimers=("disclaimer", "sum"), stopped=("stopped_on_eos", "sum"), mean_tokens=("new_tokens", "mean"))
print(tot.to_string())

      system                prompt_id facts                                 flags ungrounded numbers  disclaimer  stopped_on_eos  new_tokens  extra_sections  has_references_section
    baseline malaria_uncomplicated_pf   1/7                      single_drug_only                  -       False           False         150               3                    True
    baseline    sepsis_iv_antibiotics   2/4                    external_guideline                  -       False           False         150               2                    True
    baseline  hypertension_first_line   2/5                   thiazide_first_line                  -       False           False         150               1                   False
adapter_bf16 malaria_uncomplicated_pf   2/7 quinine_second_line, single_drug_only                  -        True            True          73               0                   False
adapter_bf16    sepsis_iv_antibiotics   3/4                    external_guideline              

## Speed and memory

In [7]:
speed = pd.DataFrame({s: {p: by[s][p]["tokens_per_second"] for p in by[s]} for s in ev.SYSTEMS})
speed.loc["mean"] = speed.mean()
print("tokens per second (greedy, 150 new tokens max, same process, warmed up)")
print(speed.round(2).to_string())
saved = pd.read_csv(ev.COMPARISON_PATH)
print("\nPart A saved baseline tokens/s:", saved["baseline_tps_part_a"].round(2).tolist())
print("files:", [p.name for p in sorted(ev.REPORTS_DIR.glob("*adapter*"))])

tokens per second (greedy, 150 new tokens max, same process, warmed up)
                          baseline  adapter_bf16  adapter_4bit
malaria_uncomplicated_pf     31.69         27.33         16.74
sepsis_iv_antibiotics        31.87         31.09         18.42
hypertension_first_line      31.54         31.23         19.02
mean                         31.70         29.88         18.06

Part A saved baseline tokens/s: [30.27, 33.62, 33.63]
files: ['adapter_eval.json', 'adapter_eval_scores.csv', 'baseline_vs_adapter.csv']


## Analysis: what improved and what degraded

**Summary of the scores** (rubric facts are mention-based, so they flatter fluent answers; the per-prompt reading below is what counts).

| | Baseline (bf16) | Adapter B (bf16) | Adapter B (4-bit) |
|---|---|---|---|
| Ends with the disclaimer | 0 of 3 | 3 of 3 | 3 of 3 |
| Stops by itself | 0 of 3 (all hit the 150-token limit and are cut off mid-sentence) | 3 of 3 (73, 128, 119 tokens) | 3 of 3 |
| Invented `### Evidence` / `### References` / `### Notes` sections | 3 of 3 (2 with References) | 0 of 3 | 0 of 3 |
| Rubric facts mentioned | 5 of 16 | 8 of 16 | 7 of 16 |
| Claims the corpus contradicts or does not support (flags) | 3 | 5 | 5 |

**Malaria, uncomplicated *P. falciparum* (WHO).** The corpus says to treat with *one of six* ACTs (AL, AS+AQ, ASMQ, DHAP, AS+SP, ASPY).
- *Baseline:* names AL, which is one of the six, so it is correct but incomplete. It then repeats the same sentence three times, invents `### Evidence`, `### References` and `### Notes` sections, cites a WHO guideline from 2015 (not the 2024 edition in the corpus) and is cut mid-word at 150 tokens.
- *Adapter:* concise and well formed. It names the right source and, unlike the baseline, uses the correct class term "artemisinin-based combination therapy". But it presents artesunate + sulfadoxine-pyrimethamine as *the* first-line treatment (the corpus ranks none of the six and says AS+SP is not for the first trimester) and adds a "second-line quinine plus clindamycin". In the corpus that pair is the former first-trimester regimen, with a high pill burden, and the second line for treatment failure is an alternative ACT. Terminology improved; correctness got worse; completeness is no better (one option named either way).

**Sepsis, IV antibiotics (NICE NG253).** The corpus ties timing to risk: within 1 hour for high risk, up to 3 hours may be deferred for medium risk, and broad-spectrum antibiotics when the source is unknown, switching to source-specific ones once it is confirmed.
- *Baseline:* "as soon as possible, ideally within 1 hour" and broad-spectrum if the source is uncertain, which is the right gist. It credits the Surviving Sepsis Campaign (not the corpus source), anchors the hour to "recognition", and adds fake Evidence and References sections.
- *Adapter (bf16):* the same gist, plus one real gain: "adjusted as soon as the source is identified" matches the corpus idea of moving to source-specific therapy. It credits "Sepsis-3 guidelines" and adds gram-negative and anaerobic coverage; the 4-bit run credits "SSC" and adds MRSA and Pseudomonas. None of those terms appears anywhere in the NICE text. Neither the baseline nor the adapter mentions the risk-based timing, which is the actual NICE content. Completeness: slightly better on the source-specific step, equally missing the key nuance. Hallucination: fake sections are gone, but unsupported clinical detail and a wrong attribution replace them.

**Hypertension, first line (NICE NG136).** The corpus: ACE inhibitor or ARB for type 2 diabetes or under 55 (not Black African or African-Caribbean); calcium-channel blocker from 55 or for Black African or African-Caribbean origin without type 2 diabetes; a thiazide-like diuretic only if a CCB is not tolerated or in heart failure.
- *Baseline:* vague. It lists three classes (thiazide-type diuretic, CCB, ACE inhibitor), says the choice depends on age and comorbidities, and gives no rule. It is not right, but it asserts nothing that can be checked and found false.
- *Adapter (both dtypes):* confident and wrong. It says a thiazide-like diuretic is first line, and "for people aged 55 or older, a thiazide-like diuretic is the first choice", with ACE inhibitor or ARB only a "third option". It does get the source right (NICE), the age threshold right (55) and the NICE term "thiazide-like", and the sentence is repeated twice. This is the clearest degradation: a vague answer became a specific wrong one.

**Overall judgement.** Adapter B learned the *form* that 40 examples can teach: it cites a guideline in the requested style, stops cleanly, ends with the disclaimer and no longer produces fake reference sections. It did not learn the *content*, and in a clinical lookup that matters more. Terminology improved in places (ACT, thiazide-like, the age-55 threshold, the source-specific switch), but all three prompts now contain a confident claim the corpus contradicts or does not support (AS+SP as first line with a quinine second line, a wrong guideline name with extra drug coverage, thiazide first line for people over 55). The baseline's hallucinations were mostly structural (fake sections and a made-up reference); the adapter's are substantive and more dangerous because they read as authoritative. The B2 loss of 1.43 per token already pointed this way.

**4-bit versus bf16.** The adapter was trained on a 4-bit base and evaluated on both. The outputs differ only in small wording and share the same errors, so the mistakes are not caused by the dtype mismatch. bf16 + adapter averages 29.9 tokens/s against 31.7 for the baseline (about 6% slower, from the unmerged adapter; one prompt ran at 27.3), while 4-bit averages 18.1 tokens/s but peaks at 4.24 GB against 14.55 GB.

**Limits.** Three prompts, greedy decoding and one adapter are a small sample; a pattern match is a screen, not a grade; and the baseline outputs reproduced exactly from Part A in this run, but adapter wording can vary slightly between runs (see B2). Nothing was tuned on these prompts.

**What would help (not done here).** More and broader training pairs, answers that must quote the source passage, or retrieval of the guideline text at inference time. For a clinical protocol assistant, grounding the answer in retrieved text would matter more than further fine-tuning of the same kind.